# Experiment 10: 95% Wilson Confidence Intervals (E2)

**Reviewer concern (R1):** §14 reports point estimates without CIs. R1 asked us to report 95% Wilson intervals on every refusal/ASR rate.

**This notebook:** consumes the existing `slm_safety_results_v3.json` (49 configurations) and produces a single aggregated JSON + CSV with point estimate + Wilson 95% CI for every reported proportion. Local-only: no GPU needed.

**Output:** `experiments/results/section14_intervals.json` and `section14_intervals.csv`.


## Setup

In [ ]:
%%capture
!pip install statsmodels pandas -q
import json
from pathlib import Path
import numpy as np
import pandas as pd
from statsmodels.stats.proportion import proportion_confint


In [ ]:
# Local fallback: if running outside Colab, point at the local results dir
try:
    from google.colab import drive
    drive.mount('/content/drive')
    RESULTS_DIR = Path('/content/drive/MyDrive/PhD/paper1-survey/experiments/results')
except ImportError:
    RESULTS_DIR = Path('experiments/results')
print('Reading from', RESULTS_DIR)
src = RESULTS_DIR / 'slm_safety_results_v3.json'
if not src.exists():
    raise FileNotFoundError(f'{src} not found.')
all_runs = json.loads(src.read_text())
print(f'Loaded {len(all_runs)} configurations')


## Wilson interval helper

In [ ]:
def wilson(k: int, n: int, alpha: float = 0.05):
    if n == 0:
        return 0.0, 0.0, 0.0
    p = k / n
    lo, hi = proportion_confint(k, n, alpha=alpha, method='wilson')
    return p, lo, hi

p, lo, hi = wilson(50, 100)
print(f'sanity: 50/100 -> p={p:.3f}, 95% CI [{lo:.3f}, {hi:.3f}]')


## Recompute proportions with Wilson CIs from the per-prompt details

Each entry carries `harmful_details` and `benign_details` lists with three label fields per prompt:
- `refused` (boolean) — the field the manuscript Table 1 actually reports;
- `guard_label` (`safe`/`unsafe`) — Llama Guard 3-1B's verdict;
- `kw_refused` (boolean) — keyword-based refusal detection.

We recompute proportions for all three signals from raw counts and attach 95% Wilson CIs.
This transparently exposes the judge-agreement story R1 asked about: on harmful prompts `refused` and `guard_label == safe` agree exactly (both = Llama Guard); on benign prompts `refused` is the more conservative signal that Table 1 reports (and differs from a naive `guard_label != safe` count by ~25–30 pp on some models).


In [ ]:
def label_counts(items):
    n = len(items)
    refused = sum(1 for d in items if d.get('refused'))
    # guard_label is sometimes 'unsafe\nS1' etc.; normalise on prefix
    def gl_norm(d):
        v = d.get('guard_label') or ''
        v = v.split('\n', 1)[0].strip().lower()
        return v
    guard_safe = sum(1 for d in items if gl_norm(d) == 'safe')
    guard_unsafe = sum(1 for d in items if gl_norm(d).startswith('unsafe'))
    kw = sum(1 for d in items if d.get('kw_refused'))
    has_labels = any('refused' in d or 'guard_label' in d or 'kw_refused' in d for d in items)
    return n, refused, guard_safe, guard_unsafe, kw, has_labels

def fmt(p, lo, hi):
    return {'p': float(p), 'ci_lo': float(lo), 'ci_hi': float(hi)}

def from_rate(rate, n):
    """Reconstruct a CI from a stored rate when per-prompt labels are unavailable."""
    if rate is None or n is None or n == 0:
        return fmt(0.0, 0.0, 0.0)
    k = round(rate * n)
    p, lo, hi = wilson(k, n)
    return fmt(p, lo, hi)

agg = {}
for key, entry in all_runs.items():
    label = entry.get('label', key)
    lang = entry.get('lang', 'en')
    quant = entry.get('quantization', 'fp16')
    h = entry.get('harmful_details') or []
    b = entry.get('benign_details') or []

    h_n, h_ref, h_gs, h_gu, h_kw, h_has = label_counts(h)
    b_n, b_ref, b_gs, b_gu, b_kw, b_has = label_counts(b)

    # If per-prompt labels are missing (multi-turn Exp 6 has only {topic, response};
    # decode-sample Exp 8 has empty lists), fall back to the stored aggregate rate
    # and reconstruct k = round(rate * n) for the CI math.
    if h_has and h_n > 0:
        reported_h = fmt(*wilson(h_ref, h_n))
        guard_h = fmt(*wilson(h_gs, h_n))
        keyword_h = fmt(*wilson(h_kw, h_n))
    else:
        n_h = entry.get('n_harmful') or h_n
        rep = from_rate(entry.get('harmful_refusal_rate'), n_h)
        reported_h = guard_h = keyword_h = rep
    if b_has and b_n > 0:
        reported_b = fmt(*wilson(b_ref, b_n))
        guard_b = fmt(*wilson(b_gu, b_n))
        keyword_b = fmt(*wilson(b_kw, b_n))
    else:
        n_b = entry.get('n_benign') or b_n
        rep = from_rate(entry.get('benign_refusal_rate'), n_b)
        reported_b = guard_b = keyword_b = rep

    agg[key] = {
        'label': label, 'lang': lang, 'quantization': quant,
        'n_harmful': entry.get('n_harmful') or h_n,
        'n_benign': entry.get('n_benign') or b_n,
        'has_perpoint_labels': bool(h_has or b_has),
        'reported': {
            'harmful_refusal': reported_h,
            'benign_refusal':  reported_b,
            'safety_score': reported_h['p'] - reported_b['p'],
        },
        'guard': {
            'harmful_refusal': guard_h,
            'benign_refusal':  guard_b,
            'safety_score': guard_h['p'] - guard_b['p'],
        },
        'keyword': {
            'harmful_refusal': keyword_h,
            'benign_refusal':  keyword_b,
            'safety_score': keyword_h['p'] - keyword_b['p'],
        },
    }

print(f'Aggregated {len(agg)} configs (3 judges × harmful/benign).')


## Manuscript-ready summary CSV

One row per config × judge with paste-ready CI strings. The 'reported' rows match the existing Table 1; the 'guard' rows are what Table 1 would look like if it used Llama Guard uniformly; the 'keyword' rows show the upper bound of refusal detection.


In [ ]:
rows = []
for key, m in agg.items():
    for judge in ('reported', 'guard', 'keyword'):
        h = m[judge]['harmful_refusal']
        b = m[judge]['benign_refusal']
        rows.append({
            'config': key, 'label': m['label'], 'lang': m['lang'],
            'quant': m['quantization'], 'judge': judge,
            'harmful_p': f'{h["p"]:.3f}',
            'harmful_ci': f'[{h["ci_lo"]:.3f}, {h["ci_hi"]:.3f}]',
            'harmful_n': m['n_harmful'],
            'benign_p': f'{b["p"]:.3f}',
            'benign_ci': f'[{b["ci_lo"]:.3f}, {b["ci_hi"]:.3f}]',
            'benign_n': m['n_benign'],
            'safety_score': f'{m[judge]["safety_score"]:.3f}',
        })
summary_df = pd.DataFrame(rows)
out_csv = RESULTS_DIR / 'section14_intervals.csv'
out_json = RESULTS_DIR / 'section14_intervals.json'
summary_df.to_csv(out_csv, index=False)
out_json.write_text(json.dumps(agg, indent=2))
print(f'Saved {out_csv} and {out_json}')
summary_df[summary_df.judge == 'reported'].head(20)
